In [44]:
!rm -rf NN-Project1/

In [45]:
!git clone https://www.github.com/yousefkoriem/NN-Project1.git

Cloning into 'NN-Project1'...
remote: Enumerating objects: 307, done.
remote: Counting objects: 100% (60/60), done.
remote: Compressing objects: 100% (40/40), done.
remote: Total 307 (delta 8), reused 57 (delta 5), pack-reused 247 (from 2)
Receiving objects: 100% (307/307), 251.50 MiB | 31.38 MiB/s, done.
Resolving deltas: 100% (68/68), done.


In [46]:
import os
os.chdir("/content/NN-Project1")

In [47]:
import keras
import tensorflow as tf
import numpy as np
import pandas as pd
from keras import layers
from keras.utils import text_dataset_from_directory
import spacy
import re
import pickle
from keras import layers, models, metrics, optimizers,callbacks

In [48]:
model = models.load_model("models/architecture/untrained_imdb_model.keras")

In [49]:
checkpoint = callbacks.ModelCheckpoint(
    filepath="models/architecture/best_imdb_model.keras",
    monitor='val_loss',
    save_best_only=True,
    verbose=1
)

In [50]:
early_stop = callbacks.EarlyStopping(
    monitor='val_loss',
    patience=8,
    restore_best_weights=True,
    verbose=1
)

In [51]:
reduce_lr = callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=3,
    min_lr=1e-6,
    verbose=1
)

In [52]:
# 1. Load the raw datasets
train_ds = tf.data.Dataset.load("data/processed/train_ds")
val_ds = tf.data.Dataset.load("data/processed/val_ds")
test_ds = tf.data.Dataset.load("data/processed/test_ds")

# 2. Define the label shape fix
def vectorize_text(text, label):
    return text, tf.expand_dims(label, -1)

# 3. Map the fix to the data
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.map(vectorize_text, num_parallel_calls=AUTOTUNE)
val_ds = val_ds.map(vectorize_text, num_parallel_calls=AUTOTUNE)
test_ds = test_ds.map(vectorize_text, num_parallel_calls=AUTOTUNE)

# 4. Apply cache and prefetch exactly ONCE at the very end
train_ds = train_ds.cache().prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)
test_ds = test_ds.cache().prefetch(buffer_size=AUTOTUNE)

In [53]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=100,
    callbacks=[checkpoint, early_stop, reduce_lr]
)

Epoch 1/100
623/625 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.6413 - f1_score: 0.6642 - loss: 0.5798
Epoch 1: val_loss improved from None to 0.31028, saving model to models/architecture/best_imdb_model.keras

Epoch 1: finished saving model to models/architecture/best_imdb_model.keras
625/625 ━━━━━━━━━━━━━━━━━━━━ 8s 10ms/step - accuracy: 0.7635 - f1_score: 0.7697 - loss: 0.4545 - val_accuracy: 0.8706 - val_f1_score: 0.8690 - val_loss: 0.3103 - learning_rate: 0.0010
Epoch 2/100
622/625 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.8942 - f1_score: 0.8952 - loss: 0.2672
Epoch 2: val_loss improved from 0.31028 to 0.30852, saving model to models/architecture/best_imdb_model.keras

Epoch 2: finished saving model to models/architecture/best_imdb_model.keras
625/625 ━━━━━━━━━━━━━━━━━━━━ 6s 10ms/step - accuracy: 0.9189 - f1_score: 0.9188 - loss: 0.2118 - val_accuracy: 0.8780 - val_f1_score: 0.8794 - val_loss: 0.3085 - learning_rate: 0.0010
Epoch 3/100
620/625 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/s

In [54]:
!mkdir results/tables

In [55]:
history_df = pd.DataFrame(history.history)
history_df.to_csv("results/tables/history.csv", index=False)

In [56]:
score = model.evaluate(test_ds)
score_df = pd.DataFrame([score], columns=["loss", "accuracy", "f1_score"])

  1/782 ━━━━━━━━━━━━━━━━━━━━ 39s 51ms/step - accuracy: 0.8750 - f1_score: 0.8571 - loss: 0.1880

782/782 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - accuracy: 0.8615 - f1_score: 0.8593 - loss: 0.3428


In [59]:
score_df.to_csv("results/tables/test_score.csv", index=False)

In [57]:
!mkdir results/models

In [58]:
model.save("results/models/final_imdb_model.keras")